# Algorithmic Trading and Quantitative Strategies
## Part 10: Statistical Arbitrage — Divergence Detection and Pairs Trading
**Dr. Ayhan Yuksel, CFA, FDP, FRM, PRM**

Bogazici University, EC581

## Table of Contents

1. [Divergence Detection Methods](#1.-Divergence-Detection-Methods)
2. [Kalman Filter for Dynamic Hedge Ratio](#2.-Kalman-Filter-for-Dynamic-Hedge-Ratio)
3. [Ornstein-Uhlenbeck Process](#3.-Ornstein-Uhlenbeck-Process)
4. [Reversal Timing](#4.-Reversal-Timing)
5. [Pairs Formation and Neutrality](#5.-Pairs-Formation-and-Neutrality)
6. [Full Pairs Trading Backtest](#6.-Full-Pairs-Trading-Backtest)
7. [Selected Topics](#7.-Selected-Topics)
8. [Exercises](#8.-Exercises)

## 1. Divergence Detection Methods

### 1.1 Price Ratio Approach

Once a cointegrated pair $(A, B)$ has been identified, the next task is to measure **how far the spread has deviated from its equilibrium** and decide when to enter and exit trades. The simplest divergence detection method uses the **price ratio**.

#### Log Ratio

The log price ratio between two assets is defined as:

$$r_t = \ln\left(\frac{P_t^A}{P_t^B}\right) = \ln P_t^A - \ln P_t^B$$

If the two prices are cointegrated, this ratio will be mean-reverting. We compute a **rolling z-score** to standardize deviations:

$$z_t = \frac{r_t - \bar{r}_t^{(L)}}{\sigma_t^{(L)}}$$

where $\bar{r}_t^{(L)}$ and $\sigma_t^{(L)}$ are the rolling mean and standard deviation computed over a lookback window of $L$ days.

#### Trading Rules

| Signal | Condition | Action |
|--------|-----------|--------|
| **Long spread** | $z_t < -z_{\text{entry}}$ | Buy A, Sell B |
| **Short spread** | $z_t > +z_{\text{entry}}$ | Sell A, Buy B |
| **Exit** | $\lvert z_t \rvert < z_{\text{exit}}$ | Close position |
| **Stop loss** | $\lvert z_t \rvert > z_{\text{stop}}$ | Close position |

Typical parameters:
- $z_{\text{entry}} = 2.0$ (entry threshold)
- $z_{\text{exit}} = 0.0$ or $0.5$ (exit threshold, mean reversion target)
- $z_{\text{stop}} = 4.0$ (stop-loss threshold to limit losses if divergence continues)

The key assumption is that the spread will **revert to its mean**, generating a profit when the z-score moves from the entry threshold back to zero.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import statsmodels.api as sm
from statsmodels.tsa.stattools import adfuller, coint
import yfinance as yf
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['font.size'] = 11

# Download data for a classic pairs trading example
pair = ['KO', 'PEP']
prices = yf.download(pair, start='2015-01-01', end='2024-12-31')['Close']
if isinstance(prices.columns, pd.MultiIndex):
    prices.columns = prices.columns.droplevel(1)
prices = prices.dropna()
print(f"Data: {len(prices)} days, {prices.index[0].date()} to {prices.index[-1].date()}")
prices.head()

In [ ]:
# --- Price Ratio Approach ---
lookback = 60

# Compute log price ratio
log_ratio = np.log(prices[pair[0]] / prices[pair[1]])

# Rolling z-score
rolling_mean = log_ratio.rolling(window=lookback).mean()
rolling_std = log_ratio.rolling(window=lookback).std()
z_score_ratio = (log_ratio - rolling_mean) / rolling_std

# --- 3-Panel Plot ---
fig, axes = plt.subplots(3, 1, figsize=(14, 12), sharex=True)

# Panel 1: Normalized prices
norm_prices = prices / prices.iloc[0] * 100
axes[0].plot(norm_prices[pair[0]], label=pair[0], linewidth=1.2)
axes[0].plot(norm_prices[pair[1]], label=pair[1], linewidth=1.2)
axes[0].set_title('Normalized Prices (Base = 100)', fontweight='bold')
axes[0].legend()
axes[0].set_ylabel('Price (indexed)')
axes[0].grid(True, alpha=0.3)

# Panel 2: Log ratio with rolling mean
axes[1].plot(log_ratio, label='Log Ratio', linewidth=1, alpha=0.8)
axes[1].plot(rolling_mean, label=f'Rolling Mean ({lookback}d)', 
             color='red', linewidth=1.5)
axes[1].set_title('Log Price Ratio: ln(KO/PEP)', fontweight='bold')
axes[1].legend()
axes[1].set_ylabel('Log Ratio')
axes[1].grid(True, alpha=0.3)

# Panel 3: Z-score with entry bands
axes[2].plot(z_score_ratio, label='Z-Score', linewidth=1, color='steelblue')
axes[2].axhline(y=2, color='red', linestyle='--', alpha=0.7, label='Entry (±2)')
axes[2].axhline(y=-2, color='red', linestyle='--', alpha=0.7)
axes[2].axhline(y=0, color='black', linestyle='-', alpha=0.4)
axes[2].fill_between(z_score_ratio.index, 2, z_score_ratio.values, 
                      where=z_score_ratio.values > 2, alpha=0.3, color='red',
                      label='Short spread zone')
axes[2].fill_between(z_score_ratio.index, -2, z_score_ratio.values, 
                      where=z_score_ratio.values < -2, alpha=0.3, color='green',
                      label='Long spread zone')
axes[2].set_title(f'Rolling Z-Score (Lookback = {lookback} days)', fontweight='bold')
axes[2].legend(loc='upper right', fontsize=9)
axes[2].set_ylabel('Z-Score')
axes[2].set_ylim(-4.5, 4.5)
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### 1.2 Cointegration Regression Approach

A more rigorous approach uses the **cointegration regression** to construct the spread. We estimate a linear relationship between the two price series:

$$P_t^A = \alpha + \beta \cdot P_t^B + \varepsilon_t$$

where:
- $\beta$ is the **hedge ratio** (number of shares of B to short per share of A),
- $\alpha$ is the intercept,
- $\varepsilon_t$ is the residual (the spread).

The spread is defined as:

$$S_t = P_t^A - \alpha - \beta \cdot P_t^B$$

If the two series are cointegrated, then $S_t$ is **stationary** and mean-reverting, which we confirm with the Augmented Dickey-Fuller (ADF) test.

#### Standardization

We standardize the spread using a rolling window:

$$z_t = \frac{S_t - \bar{S}_t^{(L)}}{\sigma_S^{(L)}}$$

**Important implementation note:** To avoid look-ahead bias, the hedge ratio $\beta$ should be estimated on the **training period** and then applied out-of-sample. Alternatively, one can use a rolling or expanding window to re-estimate $\beta$ periodically.

In [ ]:
# --- Cointegration Regression Approach ---
# Split into train (first half) and test (second half)
n = len(prices)
train_end = n // 2
train = prices.iloc[:train_end]
split_date = prices.index[train_end]

# Estimate hedge ratio via OLS on training data
X_train = sm.add_constant(train[pair[1]])
model = sm.OLS(train[pair[0]], X_train).fit()
alpha_ols = model.params.iloc[0]
beta_ols = model.params.iloc[1]
print(f"OLS Hedge Ratio: alpha = {alpha_ols:.4f}, beta = {beta_ols:.4f}")
print(f"R-squared: {model.rsquared:.4f}")

# Compute spread on full data using training-period parameters
spread_coint = prices[pair[0]] - alpha_ols - beta_ols * prices[pair[1]]
spread_mean = spread_coint.iloc[:train_end].mean()
spread_std = spread_coint.iloc[:train_end].std()

# ADF test on spread
adf_result = adfuller(spread_coint.dropna())
print(f"\nADF Test on Spread:")
print(f"  Test statistic: {adf_result[0]:.4f}")
print(f"  p-value: {adf_result[1]:.6f}")
print(f"  Critical values: {adf_result[4]}")

# Engle-Granger cointegration test
coint_stat, coint_pval, coint_crit = coint(prices[pair[0]], prices[pair[1]])
print(f"\nEngle-Granger Cointegration Test:")
print(f"  Test statistic: {coint_stat:.4f}")
print(f"  p-value: {coint_pval:.6f}")

# Rolling z-score of spread
z_spread = (spread_coint - spread_coint.rolling(60).mean()) / spread_coint.rolling(60).std()

# --- Plot ---
fig, axes = plt.subplots(2, 1, figsize=(14, 9), sharex=True)

# Panel 1: Spread with mean and ±2σ bands
axes[0].plot(spread_coint, linewidth=1, color='steelblue', label='Spread')
axes[0].axhline(y=spread_mean, color='black', linestyle='-', alpha=0.5, label='Mean')
axes[0].axhline(y=spread_mean + 2*spread_std, color='red', linestyle='--', 
                alpha=0.7, label='±2σ')
axes[0].axhline(y=spread_mean - 2*spread_std, color='red', linestyle='--', alpha=0.7)
axes[0].axvline(x=split_date, color='orange', linestyle=':', linewidth=2, 
                label='Train/Test Split')
axes[0].set_title(f'Cointegration Spread: {pair[0]} - {beta_ols:.3f}×{pair[1]}', 
                  fontweight='bold')
axes[0].legend(loc='upper right')
axes[0].set_ylabel('Spread ($)')
axes[0].grid(True, alpha=0.3)

# Panel 2: Z-score
axes[1].plot(z_spread, linewidth=1, color='steelblue')
axes[1].axhline(y=2, color='red', linestyle='--', alpha=0.7, label='±2')
axes[1].axhline(y=-2, color='red', linestyle='--', alpha=0.7)
axes[1].axhline(y=0, color='black', linestyle='-', alpha=0.4)
axes[1].axvline(x=split_date, color='orange', linestyle=':', linewidth=2, 
                label='Train/Test Split')
axes[1].set_title('Rolling Z-Score of Spread', fontweight='bold')
axes[1].set_ylabel('Z-Score')
axes[1].legend(loc='upper right')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 2. Kalman Filter for Dynamic Hedge Ratio

A fixed hedge ratio estimated via OLS assumes a **constant relationship** between the two assets. In practice, this relationship **drifts over time** due to changes in fundamentals, market regimes, or macroeconomic conditions.

The **Kalman filter** handles this by modeling the hedge ratio as a **time-varying state** that evolves according to a random walk.

### State-Space Model

**Observation equation:**

$$P_t^A = \begin{bmatrix} 1 & P_t^B \end{bmatrix} \begin{bmatrix} \alpha_t \\ \beta_t \end{bmatrix} + v_t, \qquad v_t \sim \mathcal{N}(0, V_e)$$

**State transition equation:**

$$\begin{bmatrix} \alpha_t \\ \beta_t \end{bmatrix} = \begin{bmatrix} \alpha_{t-1} \\ \beta_{t-1} \end{bmatrix} + \begin{bmatrix} w_{1,t} \\ w_{2,t} \end{bmatrix}, \qquad \mathbf{w}_t \sim \mathcal{N}(\mathbf{0}, Q)$$

The states $\alpha_t$ (intercept) and $\beta_t$ (hedge ratio) follow a **random walk**, allowing the model to adapt to structural changes.

### Kalman Filter Recursion

At each time step $t$:

1. **Predict:**
   - $\hat{\boldsymbol{\theta}}_{t|t-1} = \hat{\boldsymbol{\theta}}_{t-1|t-1}$
   - $P_{t|t-1} = P_{t-1|t-1} + Q$

2. **Update:**
   - Innovation: $e_t = P_t^A - \mathbf{H}_t' \hat{\boldsymbol{\theta}}_{t|t-1}$
   - Innovation variance: $F_t = \mathbf{H}_t' P_{t|t-1} \mathbf{H}_t + V_e$
   - Kalman gain: $K_t = P_{t|t-1} \mathbf{H}_t / F_t$
   - State update: $\hat{\boldsymbol{\theta}}_{t|t} = \hat{\boldsymbol{\theta}}_{t|t-1} + K_t e_t$
   - Covariance update: $P_{t|t} = (I - K_t \mathbf{H}_t') P_{t|t-1}$

where $\mathbf{H}_t = \begin{bmatrix} 1 \\ P_t^B \end{bmatrix}$ is the observation vector.

**Advantages over static OLS:**
- Adapts to regime changes in the hedge ratio
- Produces a time-varying spread that may be more stationary
- The prediction error $e_t$ can be used directly as the trading signal

In [ ]:
def kalman_filter_hedge_ratio(y, x, delta=1e-4, Ve=1e-3):
    """
    Kalman filter for dynamic hedge ratio estimation.
    
    Parameters
    ----------
    y : array-like
        Dependent asset prices (asset A).
    x : array-like
        Independent asset prices (asset B).
    delta : float
        State transition covariance scaling (controls adaptability).
    Ve : float
        Observation noise variance.
    
    Returns
    -------
    alpha : array
        Time-varying intercept.
    beta : array
        Time-varying hedge ratio.
    spread : array
        Kalman filter spread (prediction errors).
    Q_ratio : array
        Ratio of prediction error to its standard deviation.
    """
    n = len(y)
    
    # State: [alpha, beta]
    theta = np.zeros((n, 2))  # state estimates
    P = np.zeros((n, 2, 2))   # state covariance
    e = np.zeros(n)           # prediction errors
    Q_ratio = np.zeros(n)     # standardized errors
    
    # Initialize
    theta[0] = [0.0, 1.0]    # start with beta=1
    P[0] = np.eye(2) * 1.0   # initial uncertainty
    
    # State transition covariance (random walk)
    Q = delta * np.eye(2)
    
    for t in range(1, n):
        # Observation vector
        H = np.array([1.0, x[t]])
        
        # Predict
        theta_pred = theta[t-1].copy()
        P_pred = P[t-1] + Q
        
        # Innovation
        e[t] = y[t] - H @ theta_pred
        
        # Innovation variance
        F = H @ P_pred @ H + Ve
        
        # Standardized prediction error
        Q_ratio[t] = e[t] / np.sqrt(F)
        
        # Kalman gain
        K = P_pred @ H / F
        
        # Update
        theta[t] = theta_pred + K * e[t]
        P[t] = P_pred - np.outer(K, H) @ P_pred
    
    alpha = theta[:, 0]
    beta = theta[:, 1]
    spread = e
    
    return alpha, beta, spread, Q_ratio

# Apply Kalman filter to KO/PEP
y_vals = prices[pair[0]].values
x_vals = prices[pair[1]].values

kf_alpha, kf_beta, kf_spread, kf_qratio = kalman_filter_hedge_ratio(
    y_vals, x_vals, delta=1e-4, Ve=1e-3
)

# --- Plot Kalman Filter Results ---
fig, axes = plt.subplots(3, 1, figsize=(14, 12), sharex=True)

# Panel 1: Dynamic beta (hedge ratio)
axes[0].plot(prices.index, kf_beta, linewidth=1.2, color='purple')
axes[0].axhline(y=beta_ols, color='red', linestyle='--', alpha=0.7, 
                label=f'Static OLS β = {beta_ols:.3f}')
axes[0].set_title('Kalman Filter: Dynamic Hedge Ratio (β)', fontweight='bold')
axes[0].set_ylabel('Beta')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Panel 2: Dynamic alpha (intercept)
axes[1].plot(prices.index, kf_alpha, linewidth=1.2, color='teal')
axes[1].axhline(y=alpha_ols, color='red', linestyle='--', alpha=0.7,
                label=f'Static OLS α = {alpha_ols:.3f}')
axes[1].set_title('Kalman Filter: Dynamic Intercept (α)', fontweight='bold')
axes[1].set_ylabel('Alpha')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Panel 3: Kalman spread z-score (Q-ratio)
axes[2].plot(prices.index, kf_qratio, linewidth=0.8, color='steelblue')
axes[2].axhline(y=2, color='red', linestyle='--', alpha=0.7, label='±2')
axes[2].axhline(y=-2, color='red', linestyle='--', alpha=0.7)
axes[2].axhline(y=0, color='black', linestyle='-', alpha=0.3)
axes[2].set_title('Kalman Filter: Standardized Prediction Error', fontweight='bold')
axes[2].set_ylabel('Q-Ratio')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Kalman Beta — Mean: {kf_beta[60:].mean():.4f}, "
      f"Std: {kf_beta[60:].std():.4f}, "
      f"Final: {kf_beta[-1]:.4f}")

## 3. Ornstein-Uhlenbeck Process

The **Ornstein-Uhlenbeck (OU) process** is the continuous-time analog of a mean-reverting process. It is defined by the stochastic differential equation:

$$dX_t = \theta (\mu - X_t) \, dt + \sigma \, dW_t$$

where:
- $X_t$ is the spread at time $t$,
- $\theta > 0$ is the **speed of mean reversion** (higher $\theta$ means faster reversion),
- $\mu$ is the **long-run mean** of the spread,
- $\sigma$ is the **volatility** of the process,
- $W_t$ is a standard Brownian motion.

### Discrete-Time Calibration

The OU process can be discretized as an AR(1) model:

$$X_t - X_{t-1} = a + b \cdot X_{t-1} + \varepsilon_t$$

Running an OLS regression of $\Delta X_t$ on $X_{t-1}$ yields estimates $\hat{a}$ and $\hat{b}$, from which the OU parameters are recovered:

$$\theta = -\frac{\hat{b}}{\Delta t}, \qquad \mu = -\frac{\hat{a}}{\hat{b}}, \qquad \sigma = \frac{\hat{\sigma}_\varepsilon}{\sqrt{\Delta t}}$$

where $\Delta t = 1/252$ for daily data and $\hat{\sigma}_\varepsilon$ is the standard error of the regression residuals.

### Half-Life of Mean Reversion

The **half-life** measures how long it takes the spread to revert halfway to its mean:

$$t_{1/2} = \frac{\ln 2}{\theta}$$

For pairs trading:
- **Short half-life** (5-20 days): Highly tradeable, frequent opportunities
- **Medium half-life** (20-60 days): Moderate frequency
- **Long half-life** (>60 days): May be too slow for active trading

In [ ]:
def calibrate_ou(spread, dt=1/252):
    """
    Calibrate Ornstein-Uhlenbeck parameters from a spread series.
    
    Parameters
    ----------
    spread : array-like
        The spread series to calibrate.
    dt : float
        Time step (1/252 for daily data).
    
    Returns
    -------
    dict : OU parameters (theta, mu, sigma, half_life, half_life_days)
    """
    spread = np.array(spread)
    X = spread[:-1]
    dX = np.diff(spread)
    
    # OLS regression: dX = a + b * X_{t-1}
    X_with_const = sm.add_constant(X)
    model = sm.OLS(dX, X_with_const).fit()
    a_hat = model.params[0]
    b_hat = model.params[1]
    resid_std = np.std(model.resid)
    
    # Recover OU parameters
    theta = -b_hat / dt
    mu = -a_hat / b_hat
    sigma = resid_std / np.sqrt(dt)
    
    # Half-life in years and days
    if theta > 0:
        half_life = np.log(2) / theta
        half_life_days = half_life * 252
    else:
        half_life = np.inf
        half_life_days = np.inf
    
    return {
        'theta': theta,
        'mu': mu,
        'sigma': sigma,
        'half_life': half_life,
        'half_life_days': half_life_days,
        'b_hat': b_hat,
        'b_pvalue': model.pvalues[1],
        'r_squared': model.rsquared
    }

# Calibrate OU on the cointegration spread
ou_params = calibrate_ou(spread_coint.dropna().values)

print("Ornstein-Uhlenbeck Calibration Results")
print("=" * 45)
print(f"  Speed of reversion (θ):   {ou_params['theta']:.4f}")
print(f"  Long-run mean (μ):        {ou_params['mu']:.4f}")
print(f"  Volatility (σ):           {ou_params['sigma']:.4f}")
print(f"  Half-life (years):        {ou_params['half_life']:.4f}")
print(f"  Half-life (trading days): {ou_params['half_life_days']:.1f}")
print(f"  AR(1) coefficient (b):    {ou_params['b_hat']:.6f}")
print(f"  b p-value:                {ou_params['b_pvalue']:.6f}")
print(f"  R-squared:                {ou_params['r_squared']:.6f}")

if ou_params['theta'] > 0:
    print(f"\n→ Mean-reverting with half-life of ~{ou_params['half_life_days']:.0f} trading days.")
else:
    print("\n→ WARNING: Not mean-reverting (θ ≤ 0).")

### 3.1 OU-Based Entry and Exit Signals

The calibrated OU parameters $(\theta, \mu, \sigma)$ give us a more principled way to set entry and exit thresholds than a rolling z-score.

#### Stationary Distribution

The OU process has a known stationary (long-run) distribution:

$$X_\infty \sim N\!\left(\mu,\;\frac{\sigma^2}{2\theta}\right)$$

The equilibrium standard deviation is therefore:

$$\sigma_{eq} = \frac{\sigma}{\sqrt{2\theta}}$$

This is the natural scale of fluctuations around $\mu$. A spread that is $2\sigma_{eq}$ away from the mean is genuinely unusual; $0.5\sigma_{eq}$ is normal noise.

#### Z-Score Using OU Parameters

Instead of a rolling z-score (which depends on window length and adapts to non-stationary drift), we standardize using the calibrated parameters:

$$z_t = \frac{X_t - \mu}{\sigma_{eq}}$$

#### Trading Rules

| Signal | Condition | Action |
|---|---|---|
| Long spread | $z_t < -z_{entry}$ | Buy the spread |
| Short spread | $z_t > +z_{entry}$ | Sell the spread |
| Exit | $z_t$ crosses back to $\pm z_{exit}$ | Close position |
| Stop loss | $\lvert z_t \rvert > z_{stop}$ | Close position |

Typical values: $z_{entry} = 1.5$ to $2.0$, $z_{exit} = 0$ to $0.5$, $z_{stop} = 3.5$ to $4.0$.

#### Expected Future Path

Given the current spread value $X_0$, the conditional expectation at time $t$ is:

$$E[X_t \mid X_0] = \mu + (X_0 - \mu)\,e^{-\theta t}$$

This means if you enter when the spread is at $X_0 = \mu + 2\sigma_{eq}$, after one half-life you expect the spread to be at $\mu + \sigma_{eq}$ — capturing half the deviation as profit. This lets you estimate expected P&L before entering, which a rolling z-score cannot do.

#### Half-Life as Maximum Holding Period

The half-life $t_{1/2} = \ln 2 / \theta$ directly informs the maximum holding period. After $k$ half-lives, the expected reversion is $1 - 2^{-k}$:

| Half-lives elapsed | Expected reversion |
|---|---|
| 1 | 50% |
| 2 | 75% |
| 3 | 87.5% |

If the spread has not reverted after 2–3 half-lives, the OU model is likely misspecified and the position should be closed.

#### Limitation

All of this assumes the OU parameters are stable. In practice $\theta$, $\mu$, and $\sigma$ drift over time — which is why the Kalman filter from Section 2 is useful. A common approach is to recalibrate the OU model on a rolling window (e.g., trailing 252 days) and update thresholds accordingly.

## 4. Reversal Timing

Detecting that the spread has diverged (e.g., $|z| > 2$) is necessary but **not sufficient** for entering a trade. The spread could continue to diverge further. Several techniques can improve entry timing:

### 4.1 Bollinger Bands on the Spread

Apply Bollinger Bands directly to the spread:

$$\text{Upper Band} = \bar{S}_t^{(L)} + k \cdot \sigma_S^{(L)}, \qquad \text{Lower Band} = \bar{S}_t^{(L)} - k \cdot \sigma_S^{(L)}$$

A reversal signal occurs when the spread **touches a band and then crosses back inside** it, indicating that the divergence has peaked.

### 4.2 RSI on the Spread

The Relative Strength Index (RSI) can be applied to the spread to detect overbought/oversold conditions:

- RSI < 30 on the spread → Spread is oversold → Potential long spread entry
- RSI > 70 on the spread → Spread is overbought → Potential short spread entry

### 4.3 Waiting for First Reversal

Instead of entering immediately when $|z| > 2$, wait for the first sign of **reversal**:
- Enter long spread only when $z_t > -2$ after having been below $-2$ (i.e., z-score is moving back toward zero)
- This avoids catching a falling knife if the spread continues to diverge

### 4.4 Avoiding Divergence for a Reason

Not all divergences revert. Some represent **permanent structural shifts**. Common reasons to be cautious:

- **Post-Earnings Announcement Drift (PEAD):** If one stock just reported earnings, the spread divergence may be justified by new information and will not revert.
- **Non-Common Factor Exposure:** If the pair is diverging due to a factor that affects only one stock (e.g., regulatory change, M&A), mean reversion is unlikely.
- **Regime Breaks:** Structural changes in the business model, competitive landscape, or macroeconomic environment can permanently alter the relationship.

Always check for **recent news** or **fundamental events** before trading a divergence signal.

## 5. Pairs Formation and Neutrality

Constructing a pairs trading portfolio involves careful attention to various dimensions of **neutrality** to minimize exposure to systematic risks.

### 5.1 Dollar Neutrality

The simplest form of neutrality ensures that the **dollar value** of long and short positions are equal:

$$\text{Long notional} = \text{Short notional}$$

For a pair $(A, B)$ with hedge ratio $\beta$: for every \$1 long in A, go \$$\beta$ short in B.

### 5.2 Beta Neutrality

Dollar neutrality does not guarantee **market neutrality**. If the two stocks have different betas to the market, the portfolio will have residual market exposure. To achieve beta neutrality:

$$n_A \cdot \beta_A^{\text{mkt}} \cdot P_A + n_B \cdot \beta_B^{\text{mkt}} \cdot P_B = 0$$

where $n_A, n_B$ are the number of shares (positive for long, negative for short) and $\beta_A^{\text{mkt}}, \beta_B^{\text{mkt}}$ are the market betas.

### 5.3 Sector Neutrality

Pairs within the **same sector** naturally provide sector neutrality. Cross-sector pairs may carry unintended sector exposure. In a multi-pair portfolio, ensuring that total long and short positions are balanced across sectors reduces risk.

### 5.4 Factor Neutrality

Extending beta neutrality to multiple factors (size, value, momentum, etc.):

$$\sum_i n_i \cdot P_i \cdot \beta_i^{(k)} = 0 \qquad \text{for each factor } k$$

This is achieved by:
1. Estimating factor exposures (betas) for each stock
2. Constructing the portfolio weights to satisfy all neutrality constraints simultaneously
3. Using optimization to find weights that maximize expected return subject to neutrality constraints

**Hierarchy of neutrality** (from simplest to most complex):

| Level | Type | What it controls |
|-------|------|------------------|
| 1 | Dollar neutral | Capital allocation |
| 2 | Beta neutral | Market risk |
| 3 | Sector neutral | Sector risk |
| 4 | Factor neutral | Systematic factor risks |

## 6. Full Pairs Trading Backtest

We now implement a complete backtest for a pairs trading strategy. The strategy:
1. Computes the spread using a static or rolling hedge ratio
2. Calculates the rolling z-score
3. Generates entry and exit signals
4. Tracks positions and computes returns

In [ ]:
def pairs_trading_backtest(prices_a, prices_b, lookback=60, entry_z=2.0, 
                           exit_z=0.0, stop_z=4.0, max_holding_days=None,
                           hedge_method='rolling'):
    """
    Full pairs trading backtest.
    
    Parameters
    ----------
    prices_a, prices_b : pd.Series
        Price series for the two assets.
    lookback : int
        Lookback window for rolling hedge ratio and z-score.
    entry_z : float
        Z-score threshold for entry.
    exit_z : float
        Z-score threshold for exit (mean reversion target).
    stop_z : float
        Z-score threshold for stop loss.
    max_holding_days : int or None
        Maximum holding period. None means no limit.
    hedge_method : str
        'rolling' for rolling OLS, 'static' for full-sample OLS.
    
    Returns
    -------
    results : dict
        Dictionary with backtest results and time series.
    """
    n = len(prices_a)
    idx = prices_a.index
    
    # --- Compute hedge ratio ---
    beta = pd.Series(index=idx, dtype=float)
    alpha = pd.Series(index=idx, dtype=float)
    
    if hedge_method == 'rolling':
        for i in range(lookback, n):
            window_b = prices_b.iloc[i-lookback:i]
            window_a = prices_a.iloc[i-lookback:i]
            X = sm.add_constant(window_b.values)
            model = sm.OLS(window_a.values, X).fit()
            alpha.iloc[i] = model.params[0]
            beta.iloc[i] = model.params[1]
    else:  # static
        X_all = sm.add_constant(prices_b.values)
        model = sm.OLS(prices_a.values, X_all).fit()
        alpha[:] = model.params[0]
        beta[:] = model.params[1]
    
    # --- Compute spread and z-score ---
    spread = prices_a - alpha - beta * prices_b
    spread_mean = spread.rolling(window=lookback).mean()
    spread_std = spread.rolling(window=lookback).std()
    z_score = (spread - spread_mean) / spread_std
    
    # --- Generate signals and track positions ---
    position = pd.Series(0.0, index=idx)   # +1 long spread, -1 short spread, 0 flat
    entry_dates = []
    exit_dates = []
    trades = []
    
    holding_counter = 0
    current_pos = 0
    entry_date = None
    
    for i in range(lookback + 1, n):
        z = z_score.iloc[i]
        
        if np.isnan(z):
            position.iloc[i] = current_pos
            continue
        
        # --- Entry logic ---
        if current_pos == 0:
            if z < -entry_z:  # spread is too low → long spread (buy A, sell B)
                current_pos = 1
                entry_date = idx[i]
                entry_dates.append(entry_date)
                holding_counter = 0
            elif z > entry_z:  # spread is too high → short spread (sell A, buy B)
                current_pos = -1
                entry_date = idx[i]
                entry_dates.append(entry_date)
                holding_counter = 0
        
        # --- Exit logic ---
        elif current_pos != 0:
            holding_counter += 1
            exit_signal = False
            exit_reason = ''
            
            # Mean reversion exit
            if current_pos == 1 and z >= -exit_z:
                exit_signal = True
                exit_reason = 'mean_reversion'
            elif current_pos == -1 and z <= exit_z:
                exit_signal = True
                exit_reason = 'mean_reversion'
            
            # Stop loss
            if abs(z) > stop_z:
                exit_signal = True
                exit_reason = 'stop_loss'
            
            # Max holding period
            if max_holding_days is not None and holding_counter >= max_holding_days:
                exit_signal = True
                exit_reason = 'max_holding'
            
            if exit_signal:
                trades.append({
                    'entry_date': entry_date,
                    'exit_date': idx[i],
                    'direction': 'long_spread' if current_pos == 1 else 'short_spread',
                    'holding_days': holding_counter,
                    'exit_reason': exit_reason
                })
                exit_dates.append(idx[i])
                current_pos = 0
                holding_counter = 0
        
        position.iloc[i] = current_pos
    
    # --- Compute returns ---
    ret_a = prices_a.pct_change()
    ret_b = prices_b.pct_change()
    
    # Spread return: long A, short B (scaled by beta for dollar neutrality)
    spread_return = position.shift(1) * (ret_a - beta.shift(1) * ret_b)
    spread_return = spread_return.fillna(0)
    
    cumulative_return = (1 + spread_return).cumprod() - 1
    
    # --- Performance metrics ---
    active_returns = spread_return[spread_return != 0]
    total_return = cumulative_return.iloc[-1]
    ann_return = (1 + total_return) ** (252 / n) - 1
    ann_vol = spread_return.std() * np.sqrt(252)
    sharpe = ann_return / ann_vol if ann_vol > 0 else 0
    
    # Maximum drawdown
    wealth = (1 + spread_return).cumprod()
    peak = wealth.cummax()
    drawdown = (wealth - peak) / peak
    max_drawdown = drawdown.min()
    
    # Trade statistics
    trades_df = pd.DataFrame(trades)
    n_trades = len(trades_df)
    
    # Print summary
    print("\n" + "=" * 60)
    print("PAIRS TRADING BACKTEST RESULTS")
    print("=" * 60)
    print(f"  Strategy Parameters:")
    print(f"    Lookback: {lookback}, Entry Z: {entry_z}, Exit Z: {exit_z}")
    print(f"    Stop Z: {stop_z}, Max Holding: {max_holding_days}")
    print(f"    Hedge Method: {hedge_method}")
    print(f"  Performance:")
    print(f"    Total Return:      {total_return:>10.2%}")
    print(f"    Annualized Return: {ann_return:>10.2%}")
    print(f"    Annualized Vol:    {ann_vol:>10.2%}")
    print(f"    Sharpe Ratio:      {sharpe:>10.2f}")
    print(f"    Max Drawdown:      {max_drawdown:>10.2%}")
    print(f"  Trade Statistics:")
    print(f"    Number of trades:  {n_trades}")
    if n_trades > 0:
        print(f"    Avg holding (days): {trades_df['holding_days'].mean():.1f}")
        exit_reasons = trades_df['exit_reason'].value_counts()
        for reason, count in exit_reasons.items():
            print(f"    Exit - {reason}: {count}")
    print("=" * 60)
    
    results = {
        'spread': spread,
        'z_score': z_score,
        'position': position,
        'spread_return': spread_return,
        'cumulative_return': cumulative_return,
        'drawdown': drawdown,
        'trades': trades_df,
        'total_return': total_return,
        'ann_return': ann_return,
        'ann_vol': ann_vol,
        'sharpe': sharpe,
        'max_drawdown': max_drawdown,
        'beta': beta
    }
    
    return results

# Run backtest
bt = pairs_trading_backtest(
    prices[pair[0]], prices[pair[1]],
    lookback=60, entry_z=2.0, exit_z=0.0, stop_z=4.0,
    max_holding_days=60, hedge_method='rolling'
)

In [ ]:
# --- 4-Panel Backtest Visualization ---
fig, axes = plt.subplots(4, 1, figsize=(14, 16), sharex=True)

# Panel 1: Normalized prices
norm_p = prices / prices.iloc[0] * 100
axes[0].plot(norm_p[pair[0]], label=pair[0], linewidth=1.2)
axes[0].plot(norm_p[pair[1]], label=pair[1], linewidth=1.2)
axes[0].set_title(f'Normalized Prices: {pair[0]} vs {pair[1]}', fontweight='bold')
axes[0].legend()
axes[0].set_ylabel('Price (indexed)')
axes[0].grid(True, alpha=0.3)

# Panel 2: Z-score with entry/exit/stop bands
axes[1].plot(bt['z_score'], linewidth=0.8, color='steelblue', label='Z-Score')
axes[1].axhline(y=2, color='red', linestyle='--', alpha=0.7, label='Entry (±2)')
axes[1].axhline(y=-2, color='red', linestyle='--', alpha=0.7)
axes[1].axhline(y=4, color='darkred', linestyle=':', alpha=0.7, label='Stop (±4)')
axes[1].axhline(y=-4, color='darkred', linestyle=':', alpha=0.7)
axes[1].axhline(y=0, color='black', linestyle='-', alpha=0.3, label='Exit (0)')
axes[1].fill_between(bt['z_score'].index, 2, bt['z_score'].clip(lower=2), 
                      alpha=0.2, color='red')
axes[1].fill_between(bt['z_score'].index, -2, bt['z_score'].clip(upper=-2), 
                      alpha=0.2, color='green')
axes[1].set_title('Z-Score with Trading Bands', fontweight='bold')
axes[1].set_ylabel('Z-Score')
axes[1].legend(loc='upper right', fontsize=9)
axes[1].set_ylim(-5, 5)
axes[1].grid(True, alpha=0.3)

# Panel 3: Position
axes[2].fill_between(bt['position'].index, 0, bt['position'], 
                      where=bt['position'] > 0, alpha=0.5, color='green', 
                      label='Long spread')
axes[2].fill_between(bt['position'].index, 0, bt['position'], 
                      where=bt['position'] < 0, alpha=0.5, color='red', 
                      label='Short spread')
axes[2].set_title('Position', fontweight='bold')
axes[2].set_ylabel('Position')
axes[2].set_ylim(-1.5, 1.5)
axes[2].legend()
axes[2].grid(True, alpha=0.3)

# Panel 4: Cumulative return
axes[3].plot(bt['cumulative_return'] * 100, linewidth=1.5, color='darkblue')
axes[3].fill_between(bt['cumulative_return'].index, 0, 
                      bt['cumulative_return'].values * 100, alpha=0.15, color='blue')
axes[3].set_title('Cumulative Strategy Return', fontweight='bold')
axes[3].set_ylabel('Return (%)')
axes[3].set_xlabel('Date')
axes[3].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Print trade log (last 10)
if len(bt['trades']) > 0:
    print("\nRecent Trades (last 10):")
    print(bt['trades'].tail(10).to_string(index=False))

## 7. Selected Topics

### 7.1 Return Profile of Mean Reversion Strategies

Mean reversion strategies like pairs trading have a distinctive return profile:

- **Positive skewness is rare.** Most trades generate small, consistent profits when spreads revert, but occasional large losses occur when spreads blow out.
- **Resembles selling insurance.** The strategy collects small premiums (spread convergence) but faces tail risk (permanent divergence).
- **Return distribution:** Tends to have **negative skewness** and **fat tails** (excess kurtosis), similar to short volatility strategies.


### 7.2 Stop Losses in Mean Reversion

Stop losses in mean reversion are **conceptually paradoxical**: the more the spread diverges, the stronger the theoretical case for reversion. However, practical considerations demand stop losses:

- **Capital preservation:** A single blow-up can eliminate months of gains.
- **Model risk:** The cointegration relationship may have broken down permanently.
- **Leverage constraints:** Margin requirements increase as positions move against you.

Common approaches:
- **Z-score stop:** Exit when $|z| > z_{\text{stop}}$ (e.g., 4.0)
- **Maximum holding period:** Exit after $N$ days regardless of z-score
- **Dollar stop:** Exit when loss exceeds a fixed percentage of capital
- **Cointegration breakdown:** Re-test cointegration periodically; exit if the relationship breaks

### 7.3 Basket Trading with PCA

Instead of trading individual pairs, **Principal Component Analysis (PCA)** can be used to trade baskets:

1. Select a universe of related stocks (e.g., financial sector)
2. Compute PCA on their return series
3. **PC1** captures the common market/sector factor
4. The **residuals** after removing PC1 represent idiosyncratic deviations
5. Trade mean reversion on these residuals

$$R_i = \beta_i \cdot F_1 + \varepsilon_i$$

where $F_1$ is the first principal component (common factor) and $\varepsilon_i$ is the stock-specific residual. If $\varepsilon_i$ deviates significantly from zero, bet on reversion.

In [ ]:
# --- PCA Basket Trading ---
# Download financial sector stocks
fin_tickers = ['JPM', 'BAC', 'WFC', 'GS', 'MS', 'C', 'USB', 'PNC', 'BK', 'SCHW']
fin_prices = yf.download(fin_tickers, start='2020-01-01', end='2024-12-31')['Close']
if isinstance(fin_prices.columns, pd.MultiIndex):
    fin_prices.columns = fin_prices.columns.droplevel(1)
fin_prices = fin_prices.dropna()

# Compute returns
fin_returns = fin_prices.pct_change().dropna()

print(f"Financial sector data: {len(fin_returns)} days, {len(fin_tickers)} stocks")

# Standardize returns
ret_mean = fin_returns.mean()
ret_std = fin_returns.std()
ret_standardized = (fin_returns - ret_mean) / ret_std

# PCA
from sklearn.decomposition import PCA

pca = PCA()
pca.fit(ret_standardized)

# Explained variance
print("\nPCA Explained Variance Ratios:")
for i, var in enumerate(pca.explained_variance_ratio_[:5]):
    print(f"  PC{i+1}: {var:.4f} ({var*100:.1f}%)")
print(f"  PC1-PC3 cumulative: {pca.explained_variance_ratio_[:3].sum()*100:.1f}%")

# Factor loadings for PC1
pc1_loadings = pd.Series(pca.components_[0], index=fin_tickers)
print("\nPC1 Loadings (common factor):")
print(pc1_loadings.round(3).to_string())

# Compute residuals after removing PC1
pc1_scores = ret_standardized.values @ pca.components_[0]  # project onto PC1
pc1_contribution = np.outer(pc1_scores, pca.components_[0])  # reconstruct PC1 part
residuals = ret_standardized.values - pc1_contribution
residuals_df = pd.DataFrame(residuals, index=fin_returns.index, columns=fin_tickers)

# Cumulative residuals (for visualization)
cum_residuals = residuals_df.cumsum()

# --- Plot ---
fig, axes = plt.subplots(2, 1, figsize=(14, 10))

# Panel 1: Explained variance
cum_var = np.cumsum(pca.explained_variance_ratio_)
axes[0].bar(range(1, len(pca.explained_variance_ratio_)+1), 
            pca.explained_variance_ratio_, alpha=0.7, color='steelblue',
            label='Individual')
axes[0].plot(range(1, len(cum_var)+1), cum_var, 'ro-', markersize=6, 
             label='Cumulative')
axes[0].set_xlabel('Principal Component')
axes[0].set_ylabel('Explained Variance Ratio')
axes[0].set_title('PCA: Explained Variance of Financial Sector Returns', fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Panel 2: Cumulative residuals
for ticker in fin_tickers:
    axes[1].plot(cum_residuals[ticker], linewidth=1, label=ticker, alpha=0.8)
axes[1].axhline(y=0, color='black', linestyle='-', alpha=0.3)
axes[1].set_title('Cumulative Residuals After Removing PC1 (Common Factor)', 
                  fontweight='bold')
axes[1].set_xlabel('Date')
axes[1].set_ylabel('Cumulative Residual Return')
axes[1].legend(loc='upper left', fontsize=8, ncol=5)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Identify current trading opportunities
recent_z = (cum_residuals.iloc[-20:].mean() / cum_residuals.std())
print("\nRecent Z-Scores of Cumulative Residuals (potential opportunities):")
print(recent_z.sort_values().round(3).to_string())

## 8. Exercises

### Exercise 1: Divergence Methods Comparison
Compare the **price ratio** and **cointegration regression** approaches for detecting divergence in the KO/PEP pair:
- Implement both methods with the same lookback window (60 days)
- Plot the z-scores from both methods on the same chart
- Compute the correlation between the two z-score series
- Discuss: Under what conditions would the two methods give different signals?

### Exercise 2: OU Calibration Across Different Pairs
Select 5 different pairs (e.g., KO/PEP, XOM/CVX, JPM/BAC, MSFT/AAPL, HD/LOW) and for each:
- Estimate the cointegration regression
- Calibrate the OU process on the spread
- Report $\theta$, $\mu$, $\sigma$, and the half-life
- Rank the pairs by half-life. Which pairs are most suitable for active trading?

### Exercise 3: Parameter Sensitivity Analysis
Using the pairs trading backtest function, perform a sensitivity analysis:
- Vary `entry_z` from 1.0 to 3.0 in steps of 0.5
- Vary `lookback` from 20 to 120 in steps of 20
- Create a heatmap of Sharpe ratios for all (entry_z, lookback) combinations
- Discuss: Is there evidence of parameter overfitting? What is the most robust parameter region?

### Exercise 4: Sector Pairs Search
Download price data for 10 technology stocks. Systematically test all $\binom{10}{2} = 45$ possible pairs:
- Run the Engle-Granger cointegration test on each pair
- Filter pairs with p-value < 0.05
- For cointegrated pairs, calibrate the OU process and report the half-life
- Backtest the top 3 pairs (by shortest half-life) and compare performance

### Exercise 5: PCA Basket Trading Strategy
Extend the PCA basket analysis to a full trading strategy:
- Compute rolling 60-day PCA on financial sector returns
- Track the cumulative residual for each stock after removing the first principal component
- Generate trading signals: go long stocks with residual z-score < -2, short stocks with z-score > 2
- Backtest the strategy and compute the Sharpe ratio
- Compare with a simple pairs trading strategy on the best financial sector pair

## References

1. **Gatev, E., Goetzmann, W. N., & Rouwenhorst, K. G.** (2006). Pairs Trading: Performance of a Relative-Value Arbitrage Rule. *The Review of Financial Studies*, 19(3), 797–827.

2. **Vidyamurthy, G.** (2004). *Pairs Trading: Quantitative Methods and Analysis*. John Wiley & Sons.

3. **Elliott, R. J., Van Der Hoek, J., & Malcolm, W. P.** (2005). Pairs Trading. *Quantitative Finance*, 5(3), 271–276.

4. **Avellaneda, M., & Lee, J.-H.** (2010). Statistical Arbitrage in the US Equities Market. *Quantitative Finance*, 10(7), 761–782.